<a href="https://colab.research.google.com/github/rudalshan0412-code/attention-is-all-you-need-pytorch/blob/main/11)_Tokenizer_Vocabulary_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 프로젝트 경로 설정

from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/attention_is_all_you_need"
)

SRC_DIR = PROJECT_ROOT / "src"

PROJECT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /content/drive/MyDrive/attention_is_all_you_need
SRC_DIR: /content/drive/MyDrive/attention_is_all_you_need/src


In [ ]:
# 현재 src 구조 확인

for path in sorted(SRC_DIR.iterdir()):
    print(path.name)

__pycache__
attention.py
dataset.py
decoder.py
decoder_layer.py
encoder.py
encoder_layer.py
feed_forward.py
mask.py
multi_head_attention.py
positional_encoding.py
tokenizer.py
transformer.py
vocabulary.py


In [ ]:
# 기존 transformer.py 확인

print((SRC_DIR / "transformer.py").read_text())


import math

import torch.nn as nn

from src.positional_encoding import PositionalEncoding
from src.encoder import Encoder
from src.decoder import Decoder


class Transformer(nn.Module):
    def __init__(
        self,
        source_vocab_size,
        target_vocab_size,
        d_model,
        num_heads,
        d_ff,
        num_encoder_layers,
        num_decoder_layers,
        max_len,
        dropout=0.1,
    ):
        super().__init__()

        self.d_model = d_model

        self.source_embedding = nn.Embedding(
            source_vocab_size,
            d_model,
        )

        self.target_embedding = nn.Embedding(
            target_vocab_size,
            d_model,
        )

        self.source_positional_encoding = PositionalEncoding(
            d_model,
            max_len,
            dropout,
        )

        self.target_positional_encoding = PositionalEncoding(
            d_model,
            max_len,
            dropout,
        )

        self.encoder = En

In [ ]:
# 기존 mask.py 확인

print((SRC_DIR / "mask.py").read_text())


import torch


def create_padding_mask(
    token_ids,
    pad_idx,
):
    mask = token_ids != pad_idx

    mask = mask.unsqueeze(1).unsqueeze(2)

    return mask


def create_causal_mask(
    seq_len,
    device=None,
):
    mask = torch.ones(
        seq_len,
        seq_len,
        dtype=torch.bool,
        device=device,
    )

    mask = torch.tril(mask)

    mask = mask.unsqueeze(0).unsqueeze(0)

    return mask



In [ ]:
'''
tokenizer: 텍스트가 들어오면 쪼개어 리스트로 만드는 간단한 역할 수행

vocabulary: 쪼개진 문자열을 사전(vocab)에 매핑하여 고유한 정수번호(IDs)로 변환

dataset: 토큰화 -> 번호 부여 -> 문장의 양 끝에 <BOS>(시작), <EOS>(끝) 부착하여 Pytorch가 읽을 수 있는 tensor로 변환

collate_fn: Batch 처리를 할 때 문장마다 길이가 다른 tensor들을 모아 batch 내 최장 길이에 맞춰 PAD로 채우는 역할

이때, PAD = 0, UNK(알 수 없는 단어) = 1, BOS = 2, EOS = 3으로 ID를 고정시킴

'''

'\ntokenizer: 텍스트가 들어오면 쪼개어 리스트로 만드는 간단한 역할 수행\n\nvocabulary: 쪼개진 문자열을 사전(vocab)에 매핑하여 고유한 정수번호(IDs)로 변환\n\ndataset: 토큰화 -> 번호 부여 -> 문장의 양 끝에 <BOS>(시작), <EOS>(끝) 부착하여 Pytorch가 읽을 수 있는 tensor로 변환\n\ncollate_fn: Batch 처리를 할 때 문장마다 길이가 다른 tensor들을 모아 batch 내 최장 길이에 맞춰 PAD로 채우는 역할\n\n이때, PAD = 0, UNK(알 수 없는 단어) = 1, BOS = 2, EOS = 3으로 ID를 고정시킴\n\n'

In [ ]:
# tokenizer.py 전체 코드

import re


def tokenize(text):
    text = text.lower() # 소문자화

    tokens = re.findall( # 의미를 가지는 단어 덩어리 / 기호들만 추출
        r"\w+|[^\w\s]",
        text,
        flags=re.UNICODE,
    )

    return tokens

In [ ]:
# 코드 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/tokenizer.py

import re


def tokenize(text):
    text = text.lower()

    tokens = re.findall(
        r"\w+|[^\w\s]",
        text,
        flags=re.UNICODE,
    )

    return tokens

Overwriting /content/drive/MyDrive/attention_is_all_you_need/src/tokenizer.py


In [ ]:
# tokenizer import

from src.tokenizer import tokenize

In [ ]:
# 기본 Tokenization 테스트

text = "Hello, Transformer!"

tokens = tokenize(text)

print(tokens)

['hello', ',', 'transformer', '!']


In [ ]:
# lowercase 테스트

assert tokenize("HELLO") == ["hello"]
assert tokenize("Transformer") == ["transformer"]

print("lowercase test passed")

lowercase test passed


In [ ]:
# punctuation 테스트

print(tokenize("Hello, world!"))
print(tokenize("Transformers are powerful."))

['hello', ',', 'world', '!']
['transformers', 'are', 'powerful', '.']


In [ ]:
# vocaublary.py 전체 코드

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
BOS_TOKEN = "<BOS>"
EOS_TOKEN = "<EOS>"

# 고정 인덱스

PAD_IDX = 0
UNK_IDX = 1
BOS_IDX = 2
EOS_IDX = 3


class Vocabulary:
    def __init__(self, tokenized_sentences):
        self.itos = [
            PAD_TOKEN,
            UNK_TOKEN,
            BOS_TOKEN,
            EOS_TOKEN,
        ]

        self.stoi = {
            PAD_TOKEN: PAD_IDX,
            UNK_TOKEN: UNK_IDX,
            BOS_TOKEN: BOS_IDX,
            EOS_TOKEN: EOS_IDX,
        }

        for tokens in tokenized_sentences:
            for token in tokens:
                if token not in self.stoi:
                    index = len(self.itos) # 만약 고정 인덱스가 아닌 단어가 들어올 시, 들어오는 순서대로 인덱스를 부여함(중복인 경우 최초로 들어온 토큰을 따름)

                    self.stoi[token] = index
                    self.itos.append(token)

    def __len__(self): # 단어장에 등록된 전체 토큰의 개수 반환
        return len(self.itos)

    def token_to_id(self, token): # 문자열 토큰을 받아서 딕셔너리에 해당하는 번호를 찾아서 반환(없으면 UNK 반환)
        return self.stoi.get(
            token,
            UNK_IDX,
        )

    def id_to_token(self, index): # 정수 ID 하나를 단어로 변환
        return self.itos[index]

    def encode(self, tokens): # 단어 리스트 전체를 정수 리스트로 변환
        return [
            self.token_to_id(token)
            for token in tokens
        ]

    def decode(self, ids): # 정수 리스트 전체를 단어 리스트로 변환
        return [
            self.id_to_token(index)
            for index in ids
        ]

In [ ]:
# 코드 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/vocabulary.py

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
BOS_TOKEN = "<BOS>"
EOS_TOKEN = "<EOS>"

PAD_IDX = 0
UNK_IDX = 1
BOS_IDX = 2
EOS_IDX = 3


class Vocabulary:
    def __init__(self, tokenized_sentences):
        self.itos = [
            PAD_TOKEN,
            UNK_TOKEN,
            BOS_TOKEN,
            EOS_TOKEN,
        ]

        self.stoi = {
            PAD_TOKEN: PAD_IDX,
            UNK_TOKEN: UNK_IDX,
            BOS_TOKEN: BOS_IDX,
            EOS_TOKEN: EOS_IDX,
        }

        for tokens in tokenized_sentences:
            for token in tokens:
                if token not in self.stoi:
                    index = len(self.itos)

                    self.stoi[token] = index
                    self.itos.append(token)

    def __len__(self):
        return len(self.itos)

    def token_to_id(self, token):
        return self.stoi.get(
            token,
            UNK_IDX,
        )

    def id_to_token(self, index):
        return self.itos[index]

    def encode(self, tokens):
        return [
            self.token_to_id(token)
            for token in tokens
        ]

    def decode(self, ids):
        return [
            self.id_to_token(index)
            for index in ids
        ]

Overwriting /content/drive/MyDrive/attention_is_all_you_need/src/vocabulary.py


In [ ]:
# Vocabulary import

from src.vocabulary import (
    Vocabulary,
    PAD_TOKEN,
    UNK_TOKEN,
    BOS_TOKEN,
    EOS_TOKEN,
    PAD_IDX,
    UNK_IDX,
    BOS_IDX,
    EOS_IDX,
)

In [ ]:
# Token sentence pair 생성

source_sentences = [
    "I love cats.",
    "You like dogs.",
    "We study attention.",
    "Transformers are very powerful.",
]

target_sentences = [
    "Cats are lovely.",
    "Dogs are friendly animals.",
    "We learn attention.",
    "Transformers can process sequences.",
]

In [ ]:
# 각각 tokenize

tokenized_source_sentences = [
    tokenize(sentence)
    for sentence in source_sentences
]

tokenized_target_sentences = [
    tokenize(sentence)
    for sentence in target_sentences
]

# 확인

for tokens in tokenized_source_sentences:
    print(tokens)

print()

for tokens in tokenized_target_sentences:
    print(tokens)

['i', 'love', 'cats', '.']
['you', 'like', 'dogs', '.']
['we', 'study', 'attention', '.']
['transformers', 'are', 'very', 'powerful', '.']

['cats', 'are', 'lovely', '.']
['dogs', 'are', 'friendly', 'animals', '.']
['we', 'learn', 'attention', '.']
['transformers', 'can', 'process', 'sequences', '.']


In [ ]:
# Source / Target Vocabulary 생성

source_vocab = Vocabulary(
    tokenized_source_sentences
)

target_vocab = Vocabulary(
    tokenized_target_sentences
)

print(source_vocab is target_vocab) # False 여야함

False


In [ ]:
# Secial token index 확인

print(source_vocab.stoi[PAD_TOKEN]) # 0
print(source_vocab.stoi[UNK_TOKEN]) # 1
print(source_vocab.stoi[BOS_TOKEN]) # 2
print(source_vocab.stoi[EOS_TOKEN]) # 3

print(target_vocab.stoi[PAD_TOKEN]) # 0
print(target_vocab.stoi[UNK_TOKEN]) # 1
print(target_vocab.stoi[BOS_TOKEN]) # 2
print(target_vocab.stoi[EOS_TOKEN]) # 3

0
1
2
3
0
1
2
3


In [ ]:
# Vocabulary 전체 확인

for index, token in enumerate(source_vocab.itos):
    print(index, token)

for index, token in enumerate(target_vocab.itos):
    print(index, token)

0 <PAD>
1 <UNK>
2 <BOS>
3 <EOS>
4 i
5 love
6 cats
7 .
8 you
9 like
10 dogs
11 we
12 study
13 attention
14 transformers
15 are
16 very
17 powerful
0 <PAD>
1 <UNK>
2 <BOS>
3 <EOS>
4 cats
5 are
6 lovely
7 .
8 dogs
9 friendly
10 animals
11 we
12 learn
13 attention
14 transformers
15 can
16 process
17 sequences


In [ ]:
# Vocabulary size

print(
    "source_vocab_size:",
    len(source_vocab),
)

print(
    "target_vocab_size:",
    len(target_vocab),
)

source_vocab_size: 18
target_vocab_size: 18


In [ ]:
# token -> id

token = "love"

token_id = source_vocab.token_to_id(token)

print(token)
print(token_id)

love
5


In [ ]:
# id -> token

print(
    source_vocab.id_to_token(
        token_id
    )
)

love


In [ ]:
# encode / decode

tokens = tokenize(
    "I love cats."
)

ids = source_vocab.encode(
    tokens
)

decoded_tokens = source_vocab.decode(
    ids
)

print("tokens:")
print(tokens)

print()

print("ids:")
print(ids)

print()

print("decoded:")
print(decoded_tokens)

assert tokens == decoded_tokens

print("encode/decode test passed")

tokens:
['i', 'love', 'cats', '.']

ids:
[4, 5, 6, 7]

decoded:
['i', 'love', 'cats', '.']
encode/decode test passed


In [ ]:
# <UNK> test

unknown_ids = source_vocab.encode(
    ["this_token_does_not_exist"]
)

print(unknown_ids)

assert unknown_ids[0] == UNK_IDX

print("UNK test passed")

[1]
UNK test passed


In [ ]:
# dataset.py 전체 코드

import torch

from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

from src.vocabulary import (
    BOS_TOKEN,
    EOS_TOKEN,
    PAD_IDX,
)


class TranslationDataset(Dataset):
    def __init__(
        self,
        source_sentences,
        target_sentences,
        source_tokenizer,
        target_tokenizer,
        source_vocab,
        target_vocab,
    ):
        self.source_sentences = source_sentences
        self.target_sentences = target_sentences

        self.source_tokenizer = source_tokenizer
        self.target_tokenizer = target_tokenizer

        self.source_vocab = source_vocab
        self.target_vocab = target_vocab

    def __len__(self): # 총 문장 쌍의 개수 반환(전체 데이터의 개수와 batch를 몇번 돌아야 1 epoch인지 파악)
        return len(
            self.source_sentences
        )

    def __getitem__(self, index):
        source_tokens = self.source_tokenizer( # 토큰화
            self.source_sentences[index]
        )

        target_tokens = self.target_tokenizer( # 토큰화
            self.target_sentences[index]
        )

        source_tokens = ( # <BOS>, <EOS> 처음과 끝에 붙이기
            [BOS_TOKEN]
            + source_tokens
            + [EOS_TOKEN]
        )

        target_tokens = ( # <BOS>, <EOS> 처음과 끝에 붙이기
            [BOS_TOKEN]
            + target_tokens
            + [EOS_TOKEN]
        )

        source_ids = self.source_vocab.encode( # 인코딩(토큰 -> 정수)
            source_tokens
        )

        target_ids = self.target_vocab.encode( # 인코딩(토큰 -> 정수
            target_tokens
        )

        source_ids = torch.tensor( # Pytorch tensor로 감싸기(torch.long(64비트 정수)(정수형 텐서) 여야 인덱스를 읽어드릴 수 있음)
            source_ids,
            dtype=torch.long,
        )

        target_ids = torch.tensor( # Pytorch Tensor로 감싸기
            target_ids,
            dtype=torch.long,
        )

        return (
            source_ids, # Padding은 아직 진행하지 않음
            target_ids,
        )


def collate_fn(batch):
    source_sequences = [ # batch에서 source만 빼냄
        source_ids
        for source_ids, _ in batch
    ]

    target_sequences = [ # batch에서 target만 빼냄
        target_ids
        for _, target_ids in batch
    ]

    source_batch = pad_sequence( # 제일 긴 문장을 기준으로 padding
        source_sequences,
        batch_first=True, # shape 를 (batch_size, max_seq_len)으로 맞춰줌
        padding_value=PAD_IDX, # 0으로 padding함
    )

    target_batch = pad_sequence(
        target_sequences,
        batch_first=True,
        padding_value=PAD_IDX,
    )

    return (
        source_batch,
        target_batch,
    )

In [ ]:
# 코드 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/dataset.py

import torch

from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

from src.vocabulary import (
    BOS_TOKEN,
    EOS_TOKEN,
    PAD_IDX,
)


class TranslationDataset(Dataset):
    def __init__(
        self,
        source_sentences,
        target_sentences,
        source_tokenizer,
        target_tokenizer,
        source_vocab,
        target_vocab,
    ):
        self.source_sentences = source_sentences
        self.target_sentences = target_sentences

        self.source_tokenizer = source_tokenizer
        self.target_tokenizer = target_tokenizer

        self.source_vocab = source_vocab
        self.target_vocab = target_vocab

    def __len__(self):
        return len(
            self.source_sentences
        )

    def __getitem__(self, index):
        source_tokens = self.source_tokenizer(
            self.source_sentences[index]
        )

        target_tokens = self.target_tokenizer(
            self.target_sentences[index]
        )

        source_tokens = (
            [BOS_TOKEN]
            + source_tokens
            + [EOS_TOKEN]
        )

        target_tokens = (
            [BOS_TOKEN]
            + target_tokens
            + [EOS_TOKEN]
        )

        source_ids = self.source_vocab.encode(
            source_tokens
        )

        target_ids = self.target_vocab.encode(
            target_tokens
        )

        source_ids = torch.tensor(
            source_ids,
            dtype=torch.long,
        )

        target_ids = torch.tensor(
            target_ids,
            dtype=torch.long,
        )

        return (
            source_ids,
            target_ids,
        )


def collate_fn(batch):
    source_sequences = [
        source_ids
        for source_ids, _ in batch
    ]

    target_sequences = [
        target_ids
        for _, target_ids in batch
    ]

    source_batch = pad_sequence(
        source_sequences,
        batch_first=True,
        padding_value=PAD_IDX,
    )

    target_batch = pad_sequence(
        target_sequences,
        batch_first=True,
        padding_value=PAD_IDX,
    )

    return (
        source_batch,
        target_batch,
    )

Overwriting /content/drive/MyDrive/attention_is_all_you_need/src/dataset.py


In [ ]:
# Dataset import

from src.dataset import (
    TranslationDataset,
    collate_fn,
)

In [ ]:
# Dataset 생성

dataset = TranslationDataset(
    source_sentences=source_sentences,
    target_sentences=target_sentences,
    source_tokenizer=tokenize,
    target_tokenizer=tokenize,
    source_vocab=source_vocab,
    target_vocab=target_vocab,
)

In [ ]:
# Dataset 길이

print(
    "dataset length:",
    len(dataset),
)

dataset length: 4


In [ ]:
# dataset[0]

source_ids, target_ids = dataset[0]

print("source_ids:")
print(source_ids)

print()

print("target_ids:")
print(target_ids)

source_ids:
tensor([2, 4, 5, 6, 7, 3])

target_ids:
tensor([2, 4, 5, 6, 7, 3])


In [ ]:
# dtype 확인

print(source_ids.dtype)
print(target_ids.dtype)

import torch

assert source_ids.dtype == torch.long
assert target_ids.dtype == torch.long

print("dtype test passed")

torch.int64
torch.int64
dtype test passed


In [ ]:
# Dataset 결과 decode

print(
    source_vocab.decode(
        source_ids.tolist()
    )
)

print(
    target_vocab.decode(
        target_ids.tolist()
    )
)

['<BOS>', 'i', 'love', 'cats', '.', '<EOS>']
['<BOS>', 'cats', 'are', 'lovely', '.', '<EOS>']


In [ ]:
# BOS / EOS 검증

assert source_ids[0].item() == BOS_IDX
assert source_ids[-1].item() == EOS_IDX

assert target_ids[0].item() == BOS_IDX
assert target_ids[-1].item() == EOS_IDX

print("BOS/EOS test passed")

BOS/EOS test passed


In [ ]:
# 모든 sample 확인

for index in range(
    len(dataset)
):
    source_ids, target_ids = dataset[index]

    print(
        f"sample {index}"
    )

    print(
        "source shape:",
        source_ids.shape,
    )

    print(
        "target shape:",
        target_ids.shape,
    )

    print()

sample 0
source shape: torch.Size([6])
target shape: torch.Size([6])

sample 1
source shape: torch.Size([6])
target shape: torch.Size([7])

sample 2
source shape: torch.Size([6])
target shape: torch.Size([6])

sample 3
source shape: torch.Size([7])
target shape: torch.Size([7])



In [ ]:
# 모든 문장의 BOS/EOS 검증

for index in range(
    len(dataset)
):
    source_ids, target_ids = dataset[index]

    print(
        f"sample {index}"
    )

    print(
        "source shape:",
        source_ids.shape,
    )

    print(
        "target shape:",
        target_ids.shape,
    )

    print()

sample 0
source shape: torch.Size([6])
target shape: torch.Size([6])

sample 1
source shape: torch.Size([6])
target shape: torch.Size([7])

sample 2
source shape: torch.Size([6])
target shape: torch.Size([6])

sample 3
source shape: torch.Size([7])
target shape: torch.Size([7])



In [ ]:
# Padding 전 sample 3개

batch = [
    dataset[0],
    dataset[1],
    dataset[2],
]

for source_ids, target_ids in batch:
    print(
        source_ids.shape,
        target_ids.shape,
    )

torch.Size([6]) torch.Size([6])
torch.Size([6]) torch.Size([7])
torch.Size([6]) torch.Size([6])


In [ ]:
# collate_fn 테스트

source_batch, target_batch = collate_fn(
    batch
)

print("source_batch:")
print(source_batch)

print()

print("target_batch:")
print(target_batch)

source_batch:
tensor([[ 2,  4,  5,  6,  7,  3],
        [ 2,  8,  9, 10,  7,  3],
        [ 2, 11, 12, 13,  7,  3]])

target_batch:
tensor([[ 2,  4,  5,  6,  7,  3,  0],
        [ 2,  8,  5,  9, 10,  7,  3],
        [ 2, 11, 12, 13,  7,  3,  0]])


In [ ]:
# Batch Shape 확인

print(
    "source_batch.shape:",
    source_batch.shape,
)

print(
    "target_batch.shape:",
    target_batch.shape,
)

source_batch.shape: torch.Size([3, 6])
target_batch.shape: torch.Size([3, 7])


In [ ]:
# PAD 확인

print(
    "PAD_IDX:",
    PAD_IDX,
)

# 자동 검증

assert PAD_IDX == 0

assert (
    source_batch == PAD_IDX
).any() or (
    target_batch == PAD_IDX
).any()

print("padding test passed")

PAD_IDX: 0
padding test passed


In [ ]:
# 기존 Mask와 연결

from src.mask import (
    create_padding_mask,
    create_causal_mask,
)

In [ ]:
# Source Padding Mask

source_mask = create_padding_mask(
    source_batch,
    PAD_IDX,
)

print(
    "source_mask.shape:",
    source_mask.shape,
)

assert source_mask.shape == (
    source_batch.size(0),
    1,
    1,
    source_batch.size(1),
)

source_mask.shape: torch.Size([3, 1, 1, 6])


In [ ]:
# Target Padding Mask

target_padding_mask = create_padding_mask(
    target_batch,
    PAD_IDX,
)

print(
    "target_padding_mask.shape:",
    target_padding_mask.shape,
)

target_padding_mask.shape: torch.Size([3, 1, 1, 7])


In [ ]:
# Casual Mask

causal_mask = create_causal_mask(
    target_batch.size(1),
)

print(
    "causal_mask.shape:",
    causal_mask.shape,
)

causal_mask.shape: torch.Size([1, 1, 7, 7])


In [ ]:
# Target Mask 결합

target_mask = (
    target_padding_mask
    & causal_mask
)

print(
    "target_mask.shape:",
    target_mask.shape,
)

assert target_mask.shape == (
    target_batch.size(0),
    1,
    target_batch.size(1),
    target_batch.size(1),
)

target_mask.shape: torch.Size([3, 1, 7, 7])


In [ ]:
# 기존 transformer import

from src.transformer import Transformer

In [ ]:
# Transformer 생성

d_model = 8
num_heads = 2
d_ff = 32

num_encoder_layers = 2
num_decoder_layers = 2

max_len = 50

transformer = Transformer(
    source_vocab_size=len(source_vocab),
    target_vocab_size=len(target_vocab),
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_encoder_layers=num_encoder_layers,
    num_decoder_layers=num_decoder_layers,
    max_len=max_len,
    dropout=0.0,
)

In [ ]:
# Transformer forward

(
    logits,
    encoder_weights,
    decoder_self_weights,
    decoder_cross_weights,
) = transformer(
    source_batch,
    target_batch,
    source_mask,
    target_mask,
)

In [ ]:
# Logits shape

print(
    "logits.shape:",
    logits.shape,
)

assert logits.shape == (
    target_batch.size(0),
    target_batch.size(1),
    len(target_vocab),
)

print("Transformer logits shape passed")

logits.shape: torch.Size([3, 7, 18])
Transformer logits shape passed


In [ ]:
# Encoder Attention 확인

print(
    "number of encoder attention weights:",
    len(encoder_weights),
)

for index, weights in enumerate(
    encoder_weights
):
    print(
        f"encoder layer {index}:",
        weights.shape,
    )

assert len(
    encoder_weights
) == num_encoder_layers

number of encoder attention weights: 2
encoder layer 0: torch.Size([3, 2, 6, 6])
encoder layer 1: torch.Size([3, 2, 6, 6])


In [ ]:
# Decoder Self-Attention 확인

print(
    "number of decoder self attention weights:",
    len(decoder_self_weights),
)

for index, weights in enumerate(
    decoder_self_weights
):
    print(
        f"decoder layer {index}:",
        weights.shape,
    )

assert len(
    decoder_self_weights
) == num_decoder_layers

number of decoder self attention weights: 2
decoder layer 0: torch.Size([3, 2, 7, 7])
decoder layer 1: torch.Size([3, 2, 7, 7])


In [ ]:
# Decoder Cross-Attention 확인

print(
    "number of decoder cross attention weights:",
    len(decoder_cross_weights),
)

for index, weights in enumerate(
    decoder_cross_weights
):
    print(
        f"decoder layer {index}:",
        weights.shape,
    )

# 검증

assert len(
    decoder_cross_weights
) == num_decoder_layers

number of decoder cross attention weights: 2
decoder layer 0: torch.Size([3, 2, 7, 6])
decoder layer 1: torch.Size([3, 2, 7, 6])


In [ ]:
# 최종 통합 테스트

import torch

from src.tokenizer import tokenize

from src.vocabulary import (
    Vocabulary,
    PAD_IDX,
    UNK_IDX,
    BOS_IDX,
    EOS_IDX,
)

from src.dataset import (
    TranslationDataset,
    collate_fn,
)

from src.mask import (
    create_padding_mask,
    create_causal_mask,
)

from src.transformer import Transformer


# -------------------------------------------------
# 1. Special Token
# -------------------------------------------------

assert PAD_IDX == 0
assert UNK_IDX == 1
assert BOS_IDX == 2
assert EOS_IDX == 3


# -------------------------------------------------
# 2. Tokenizer
# -------------------------------------------------

assert tokenize(
    "Hello, world!"
) == [
    "hello",
    ",",
    "world",
    "!",
]


# -------------------------------------------------
# 3. Toy Data
# -------------------------------------------------

source_sentences = [
    "I love cats.",
    "You like dogs.",
    "We study attention.",
    "Transformers are very powerful.",
]

target_sentences = [
    "Cats are lovely.",
    "Dogs are friendly animals.",
    "We learn attention.",
    "Transformers can process sequences.",
]


# -------------------------------------------------
# 4. Vocabulary
# -------------------------------------------------

tokenized_source_sentences = [
    tokenize(sentence)
    for sentence in source_sentences
]

tokenized_target_sentences = [
    tokenize(sentence)
    for sentence in target_sentences
]

source_vocab = Vocabulary(
    tokenized_source_sentences
)

target_vocab = Vocabulary(
    tokenized_target_sentences
)

assert source_vocab is not target_vocab

assert source_vocab.encode(
    ["this_token_does_not_exist"]
)[0] == UNK_IDX


# -------------------------------------------------
# 5. Dataset
# -------------------------------------------------

dataset = TranslationDataset(
    source_sentences=source_sentences,
    target_sentences=target_sentences,
    source_tokenizer=tokenize,
    target_tokenizer=tokenize,
    source_vocab=source_vocab,
    target_vocab=target_vocab,
)

assert len(dataset) == len(
    source_sentences
)


# -------------------------------------------------
# 6. Dataset Items
# -------------------------------------------------

for index in range(
    len(dataset)
):
    source_ids, target_ids = dataset[index]

    assert source_ids.dtype == torch.long
    assert target_ids.dtype == torch.long

    assert source_ids[0].item() == BOS_IDX
    assert source_ids[-1].item() == EOS_IDX

    assert target_ids[0].item() == BOS_IDX
    assert target_ids[-1].item() == EOS_IDX


# -------------------------------------------------
# 7. Batch Padding
# -------------------------------------------------

batch = [
    dataset[0],
    dataset[1],
    dataset[2],
]

source_batch, target_batch = collate_fn(
    batch
)

B = len(batch)
S = source_batch.size(1)
T = target_batch.size(1)

assert source_batch.shape == (
    B,
    S,
)

assert target_batch.shape == (
    B,
    T,
)


# -------------------------------------------------
# 8. Masks
# -------------------------------------------------

source_mask = create_padding_mask(
    source_batch,
    PAD_IDX,
)

target_padding_mask = create_padding_mask(
    target_batch,
    PAD_IDX,
)

causal_mask = create_causal_mask(
    target_batch.size(1),
)

target_mask = (
    target_padding_mask
    & causal_mask
)

assert source_mask.shape == (
    B,
    1,
    1,
    S,
)

assert target_padding_mask.shape == (
    B,
    1,
    1,
    T,
)

assert causal_mask.shape == (
    1,
    1,
    T,
    T,
)

assert target_mask.shape == (
    B,
    1,
    T,
    T,
)


# -------------------------------------------------
# 9. Transformer
# -------------------------------------------------

d_model = 8
num_heads = 2
d_ff = 32

num_encoder_layers = 2
num_decoder_layers = 2

transformer = Transformer(
    source_vocab_size=len(source_vocab),
    target_vocab_size=len(target_vocab),
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_encoder_layers=num_encoder_layers,
    num_decoder_layers=num_decoder_layers,
    max_len=50,
    dropout=0.0,
)

(
    logits,
    encoder_weights,
    decoder_self_weights,
    decoder_cross_weights,
) = transformer(
    source_batch,
    target_batch,
    source_mask,
    target_mask,
)


# -------------------------------------------------
# 10. Transformer Output
# -------------------------------------------------

assert logits.shape == (
    B,
    T,
    len(target_vocab),
)

assert len(
    encoder_weights
) == num_encoder_layers

assert len(
    decoder_self_weights
) == num_decoder_layers

assert len(
    decoder_cross_weights
) == num_decoder_layers


# -------------------------------------------------
# Result
# -------------------------------------------------

print("All Stage 11 tests passed.")

print()
print(
    "source_vocab_size:",
    len(source_vocab),
)

print(
    "target_vocab_size:",
    len(target_vocab),
)

print()
print(
    "source_batch.shape:",
    source_batch.shape,
)

print(
    "target_batch.shape:",
    target_batch.shape,
)

print()
print(
    "source_mask.shape:",
    source_mask.shape,
)

print(
    "target_mask.shape:",
    target_mask.shape,
)

print()
print(
    "logits.shape:",
    logits.shape,
)

All Stage 11 tests passed.

source_vocab_size: 18
target_vocab_size: 18

source_batch.shape: torch.Size([3, 6])
target_batch.shape: torch.Size([3, 7])

source_mask.shape: torch.Size([3, 1, 1, 6])
target_mask.shape: torch.Size([3, 1, 7, 7])

logits.shape: torch.Size([3, 7, 18])
